# SkyTNT MIDI Model — Transformer Music Generation

The SkyTNT midi-model is an open-source transformer for multi-instrument MIDI generation. It can generate piano, guitar, drums, bass, and other instruments with realistic timing and dynamics.

Repository: https://github.com/SkyTNT/midi-model

In [ ]:
!git clone https://github.com/SkyTNT/midi-model.git
%cd midi-model
!pip install -r requirements.txt
!pip install pretty_midi matplotlib IPython

## Load Pre-trained Model

In [ ]:
import torch
import sys
sys.path.append('.')

try:
    from midi_model import MIDIModel
    model = MIDIModel.from_pretrained("skytnt/midi-model")
    model = model.to("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Model loaded on {next(model.parameters()).device}")
except Exception as e:
    print(f"Could not load model: {e}")
    print("Falling back to demonstration mode with pre-generated examples")
    model = None

## Unconditional Generation

Generate a piece from scratch — the model chooses instruments, melody, harmony, and rhythm.

In [ ]:
if model is not None:
    with torch.no_grad():
        output = model.generate(max_length=1024, temperature=1.0, top_k=50)
    midi_data = model.tokens_to_midi(output)
    midi_data.write('unconditional.mid')
    print(f"Generated {len(midi_data.instruments)} instruments, "
          f"{sum(len(i.notes) for i in midi_data.instruments)} total notes")
else:
    # Create a demo MIDI file manually
    import pretty_midi
    midi_data = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0, name='Demo Piano')
    import numpy as np
    np.random.seed(42)
    t = 0
    for _ in range(64):
        pitch = np.random.choice([60,62,64,65,67,69,71,72])
        dur = np.random.choice([0.25, 0.5, 1.0])
        piano.notes.append(pretty_midi.Note(velocity=80, pitch=pitch, start=t, end=t+dur*0.9))
        t += dur
    midi_data.instruments.append(piano)
    midi_data.write('unconditional.mid')
    print("Demo mode: created simple example MIDI")

## Visualize Piano Roll

In [ ]:
import pretty_midi
import matplotlib.pyplot as plt

midi = pretty_midi.PrettyMIDI('unconditional.mid')
piano_roll = midi.get_piano_roll(fs=20)

plt.figure(figsize=(15, 5))
plt.imshow(piano_roll[36:84], aspect='auto', origin='lower', cmap='magma',
           extent=[0, piano_roll.shape[1]/20, 36, 84])
plt.xlabel('Time (s)')
plt.ylabel('MIDI Note')
plt.title('Generated MIDI — Piano Roll')
plt.colorbar(label='Velocity')
plt.tight_layout()
plt.show()

# Print instrument breakdown
for inst in midi.instruments:
    print(f"  {inst.name or 'Track'}: {len(inst.notes)} notes, "
          f"program {inst.program}, {'drums' if inst.is_drum else 'melodic'}")

## Conditional Generation (Prompt)

Provide a short seed and let the model continue.

In [ ]:
# Create a seed melody
import pretty_midi
seed = pretty_midi.PrettyMIDI()
piano = pretty_midi.Instrument(program=0)
seed_notes = [(60,0,0.5), (64,0.5,1.0), (67,1.0,1.5), (72,1.5,2.0)]
for pitch, start, end in seed_notes:
    piano.notes.append(pretty_midi.Note(velocity=80, pitch=pitch, start=start, end=end))
seed.instruments.append(piano)
seed.write('seed.mid')

if model is not None:
    seed_tokens = model.midi_to_tokens(seed)
    with torch.no_grad():
        output = model.generate(prompt=seed_tokens, max_length=1024, temperature=0.9)
    result = model.tokens_to_midi(output)
    result.write('continued.mid')
    print("Generated continuation from seed")
else:
    print("Demo mode: model not loaded, skipping conditional generation")

## Experiment

1. Try different temperatures (0.5 = conservative, 1.5 = adventurous)
2. Generate with different max_length values
3. Try different seed melodies
4. Compare multi-instrument vs piano-only generation